In [6]:
import xgboost as xgb
print(xgb.__version__)

3.0.4


In [7]:
import sys, xgboost as xgb
print(sys.executable)        # should point to .../.venv/bin/python
print(xgb.__version__)       # should print 3.0.4
print(xgb.__file__)          # should live under .../.venv/...

c:\Users\hamza\Downloads\PROJET-FULL-STACK-DATA\housing-mle\.venv\Scripts\python.exe
3.0.4
c:\Users\hamza\Downloads\PROJET-FULL-STACK-DATA\housing-mle\.venv\Lib\site-packages\xgboost\__init__.py


In [8]:
# ==============================================
# 1. Imports
# ==============================================
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import optuna
import mlflow
import mlflow.xgboost

# Lightweight configuration for a mid-range Windows laptop
# Keep the search space smaller and use a limited number of CPU cores
N_TRIALS = 5
N_JOBS = 2
N_ESTIMATORS_MIN = 100
N_ESTIMATORS_MAX = 400
MAX_DEPTH_MIN = 3
MAX_DEPTH_MAX = 7

In [9]:
# ==============================================
# 2. Load processed datasets
# ==============================================
train_df = pd.read_csv(r"C:\Users\hamza\Downloads\PROJET-FULL-STACK-DATA\housing-mle\data\processed\train_final.csv")
eval_df  = pd.read_csv(r"C:\Users\hamza\Downloads\PROJET-FULL-STACK-DATA\housing-mle\data\processed\eval_final.csv")


# Define target + features
target = "price"
X_train, y_train = train_df.drop(columns=[target]), train_df[target]
X_eval, y_eval   = eval_df.drop(columns=[target]), eval_df[target]

print("Train shape:", X_train.shape)
print("Eval shape:", X_eval.shape)

Train shape: (579123, 39)
Eval shape: (146667, 39)


In [11]:
# ==============================================
# 3. Define Optuna objective function with MLflow
# ==============================================
# This objective is intentionally lightweight:
# - smaller search space
# - limited CPU usage
# - fewer trials to avoid freezing a modest laptop

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", N_ESTIMATORS_MIN, N_ESTIMATORS_MAX),
        "max_depth": trial.suggest_int("max_depth", MAX_DEPTH_MIN, MAX_DEPTH_MAX),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 6),
        "gamma": trial.suggest_float("gamma", 0.0, 3.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 1.0, log=True),
        "random_state": 42,
        "n_jobs": N_JOBS,
        "tree_method": "hist",
    }

    with mlflow.start_run(nested=True):
        model = XGBRegressor(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_eval)
        rmse = float(np.sqrt(mean_squared_error(y_eval, y_pred)))
        mae = float(mean_absolute_error(y_eval, y_pred))
        r2 = float(r2_score(y_eval, y_pred))

        # Log hyperparameters and metrics for traceability
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

    return rmse

In [15]:
# ==============================================
# 4. Run Optuna study with MLflow
# ==============================================
# Force MLflow to always use the root project mlruns folder
mlflow.set_tracking_uri(r"C:\Users\hamza\Downloads\PROJET-FULL-STACK-DATA\housing-mle\mlruns")
mlflow.set_experiment("xgboost_optuna_housing")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15)

print("Best params:", study.best_trial.params)

c:\Users\hamza\Downloads\PROJET-FULL-STACK-DATA\housing-mle\.venv\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/04/22 12:08:39 INFO mlflow.tracking.fluent: Experiment with name 'xgboost_optuna_housing' does not exist. Creating a new experiment.
[I 2026-04-22 12:08:39,332] A new study created in memory with name: no-name-2ad1b0ba-dbef-437b-9ed1-f60ab43c6420
[I 2026-04-22 12:08:53,835] Trial 0 finished with value: 48196.21076440352 and parameters: {'n_estimators': 144, 'max_depth': 5, 'learning_rate': 0.0824662995757985, 'subsample': 0.8931217283884504, 'colsample_bytree': 0.824896255488721, 'min_child_weigh

Best params: {'n_estimators': 167, 'max_depth': 7, 'learning_rate': 0.16340044858438565, 'subsample': 0.9728244154308643, 'colsample_bytree': 0.8762163152198372, 'min_child_weight': 3, 'gamma': 2.2062852822138197, 'reg_alpha': 8.518266938035326e-07, 'reg_lambda': 1.8924039310972324e-05}


In [17]:
# ==============================================
# 5. Train final model with best params and log to MLflow
# ==============================================
best_params = study.best_trial.params
best_model = XGBRegressor(**best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_eval)

mae = mean_absolute_error(y_eval, y_pred)
rmse = np.sqrt(mean_squared_error(y_eval, y_pred))
r2 = r2_score(y_eval, y_pred)

print("Final tuned model performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

# Log final model
with mlflow.start_run(run_name="best_xgboost_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    mlflow.xgboost.log_model(
        xgb_model=best_model.get_booster(),
        artifact_path="model"
    )

2026/04/22 12:12:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Final tuned model performance:
MAE: 26909.902924524282
RMSE: 44209.58820976707
R²: 0.9694423749845481


2026/04/22 12:13:01 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
